In [ ]:
"""
Long Short-Term Memory (LSTM) is a type of Recurrent Neural Network (RNN) designed to learn short-term and long-term dependencies in sequential data such as text,
speech, and time series. Unlike a standard RNN, an LSTM has both a Hidden State and a Cell State, allowing it to preserve important information over longer sequences.
It uses three main gates—Forget Gate, Input Gate, and Output Gate—to control what information should be forgotten, stored, and passed to the next time step.
This architecture helps LSTMs reduce the vanishing gradient problem compared with standard RNNs.
"""

"""
LSTM reduces the vanishing gradient problem by using a Cell State and a set of gates that control the flow of information. In a standard RNN,
during Backpropagation Through Time, gradients must pass through many recurrent steps and are repeatedly multiplied by weights and activation derivatives,
which can cause them to become extremely small. In an LSTM, the Cell State provides a relatively direct path for information and gradients across time.
When the Forget Gate has a value close to 1, the previous Cell State can be preserved with little modification, allowing gradients to flow more easily across
many time steps and reducing the likelihood of vanishing.
"""

In [2]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
# NLTK

nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /home/soheil/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/soheil/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
# Load Dataset
data = pd.read_csv("Clothing-Review.csv")

# Remove rows without Class Name
data = data[data["Class Name"].notna()].copy()

In [5]:
# Create Binary Target

def filter_score(rating):
    return int(rating > 3)


y = data["Rating"].apply(filter_score)

features = ["Class Name", "Title", "Review Text"]
X = data[features].copy()



In [6]:
# Text Preprocessing

def clean_text(text):

    # Handle missing values
    if pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)


X["Title"] = X["Title"].apply(clean_text)
X["Review Text"] = X["Review Text"].apply(clean_text)
X["Class Name"] = X["Class Name"].apply(clean_text)

In [7]:
# Combine Text Features

X["Text"] = (
    X["Class Name"] + " " +
    X["Title"] + " " +
    X["Review Text"]
)


In [8]:
#  Train / Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X["Text"],
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [9]:
#  Tokenization

MAX_VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 40

tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    oov_token="<OOV>"
)

# IMPORTANT:
# Fit tokenizer only on training data
tokenizer.fit_on_texts(X_train)

In [10]:
#  Convert Text → Integer Sequences

train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

In [11]:
#  Padding

X_train_pad = pad_sequences(
    train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)


print("Train shape:", X_train_pad.shape)
print("Test shape:", X_test_pad.shape)


Train shape: (17604, 40)
Test shape: (5868, 40)


In [13]:
# Build LSTM Model

model = keras.Sequential([

    # Convert token IDs → dense vectors
    keras.layers.Embedding(
        input_dim=MAX_VOCAB_SIZE,
        output_dim=128
    ),

    # First LSTM
    keras.layers.LSTM(
        64,
        return_sequences=True
    ),

    # Second LSTM
    keras.layers.LSTM(
        64
    ),

    # Fully Connected layer
    keras.layers.Dense(
        128,
        activation="relu"
    ),

    # Prevent overfitting
    keras.layers.Dropout(0.4),

    # Binary classification
    keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

In [15]:
#  Model Summary

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
#  Compile

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [17]:
# Train

history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/5
441/441 ━━━━━━━━━━━━━━━━━━━━ 23s 47ms/step - accuracy: 0.8472 - loss: 0.3590 - val_accuracy: 0.8577 - val_loss: 0.3170
Epoch 2/5
441/441 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - accuracy: 0.9057 - loss: 0.2468 - val_accuracy: 0.8836 - val_loss: 0.2775
Epoch 3/5
441/441 ━━━━━━━━━━━━━━━━━━━━ 19s 44ms/step - accuracy: 0.9317 - loss: 0.1897 - val_accuracy: 0.8787 - val_loss: 0.2940
Epoch 4/5
441/441 ━━━━━━━━━━━━━━━━━━━━ 20s 46ms/step - accuracy: 0.9470 - loss: 0.1508 - val_accuracy: 0.8685 - val_loss: 0.4058
Epoch 5/5
441/441 ━━━━━━━━━━━━━━━━━━━━ 19s 44ms/step - accuracy: 0.9587 - loss: 0.1214 - val_accuracy: 0.8713 - val_loss: 0.4204


In [18]:
# Evaluate on Test Set

test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=1
)

print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


184/184 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8645 - loss: 0.4087

Test Loss: 0.4086914360523224
Test Accuracy: 0.8645194172859192
Test Accuracy: 86.45%


In [19]:
#  Make Predictions

predictions = model.predict(X_test_pad)

predicted_classes = (
    predictions >= 0.5
).astype(int).flatten()

184/184 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step


In [20]:
# Show Some Predictions

for i in range(10):

    print(
        f"Actual: {y_test.iloc[i]} | "
        f"Predicted: {predicted_classes[i]} | "
        f"Probability: {predictions[i][0]:.4f}"
    )

Actual: 1 | Predicted: 1 | Probability: 0.9689
Actual: 1 | Predicted: 1 | Probability: 0.5868
Actual: 1 | Predicted: 1 | Probability: 0.9956
Actual: 1 | Predicted: 1 | Probability: 0.9991
Actual: 1 | Predicted: 1 | Probability: 0.9946
Actual: 1 | Predicted: 1 | Probability: 0.8503
Actual: 1 | Predicted: 1 | Probability: 0.9977
Actual: 1 | Predicted: 1 | Probability: 0.9391
Actual: 1 | Predicted: 1 | Probability: 0.9996
Actual: 1 | Predicted: 1 | Probability: 0.9991
